In [1]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [2]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [3]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [4]:
# Add Compounds
cnames = ["Water"]
sim.AddCompound("Water")

In [5]:
one = sim.AddFlowsheetObject('Material Stream','1')
two = sim.AddFlowsheetObject('Material Stream','2')
three = sim.AddFlowsheetObject('Material Stream','3')
four = sim.AddFlowsheetObject('Material Stream','4')
HT_1 = sim.AddFlowsheetObject('Heater','HT-1')
CL_1 = sim.AddFlowsheetObject('Cooler','CL-1')
E_1 = sim.AddFlowsheetObject('Energy Stream','E1')
E_2 = sim.AddFlowsheetObject('Energy Stream','E2')
SP_1 = sim.AddFlowsheetObject('Specification Block','SP-1')

In [6]:
one = one.GetAsObject()
two = two.GetAsObject()
three = three.GetAsObject()
four = four.GetAsObject()
HT_1 = HT_1.GetAsObject()
CL_1 = CL_1.GetAsObject()
E_1 = E_1.GetAsObject()
E_2 = E_2.GetAsObject()
SP_1 = SP_1.GetAsObject()

In [7]:
sim.ConnectObjects(one.GraphicObject , HT_1.GraphicObject, -1, -1)
sim.ConnectObjects(E_1.GraphicObject, HT_1.GraphicObject, -1, -1)
sim.ConnectObjects(HT_1.GraphicObject , two.GraphicObject, -1, -1)
sim.ConnectObjects(three.GraphicObject , CL_1.GraphicObject, -1, -1)
sim.ConnectObjects(CL_1.GraphicObject , E_2.GraphicObject, -1, -1)
sim.ConnectObjects(CL_1.GraphicObject , four.GraphicObject, -1, -1)

In [8]:
one.SetOverallComposition(Array[float]([1]))
one.SetTemperature(298.15) # K
one.SetPressure(101325.0) # Pa
one.SetMassFlow(1.0) # kg/s
three.SetOverallComposition(Array[float]([1]))
three.SetTemperature(298.15) # K
three.SetPressure(101325.0) # Pa
three.SetMassFlow(1.0) # kg/s

'3: mass flow set to 1 kg/s'

In [9]:
# property package
Steam_table = sim.CreateAndAddPropertyPackage("Steam Tables (IAPWS-IF97)")

In [10]:
# Calc Modes
HT_1.CalcMode = HT_1.CalcMode.HeatAdded
HT_1.set_DeltaQ(100)  # W
 
CL_1.CalcMode = CL_1.CalcMode.HeatRemoved
# CL_1's DeltaQ is intentionally left unset here -- it is supplied
# at solve time by the Spec block (SP-1) below.

In [11]:
# Setting up the controller block to control the pump's power requirement based on the outlet pressure of the pump. The controller will adjust the pump's power to maintain a desired outlet pressure.
from DWSIM.UnitOperations.SpecialOps.Helpers import SpecialOpObjectInfo
from DWSIM.Interfaces.Enums import SpecCalcMode2, SpecVarType

# ---------------------------------------------------------------------------
# Spec block: pushes HT-1's Heat Added onto CL-1's Heat Removed
# ---------------------------------------------------------------------------
src_info = SpecialOpObjectInfo()
src_info.ID = HT_1.Name
src_info.Name = HT_1.GraphicObject.Tag
src_info.PropertyName = "PROP_HT_3"   # Heat Added
 
tgt_info = SpecialOpObjectInfo()
tgt_info.ID = CL_1.Name
tgt_info.Name = CL_1.GraphicObject.Tag
tgt_info.PropertyName = "PROP_CL_3"   # Heat Removed
 
SP_1.set_SourceObject(HT_1)
SP_1.set_SourceObjectData(src_info)
SP_1.set_TargetObject(CL_1)
SP_1.set_TargetObjectData(tgt_info)
 
SP_1.set_Expression("X")   # target = source, 1:1 pass-through
SP_1.set_CalculateTargetObject(True)
SP_1.set_SpecCalculationMode(SpecCalcMode2.GlobalSetting)
SP_1.GraphicObject.Active = True
 
# Reverse-link the Spec block onto the objects it connects.
# Without this, DWSIM never knows to invoke SP-1 when calculating HT-1/CL-1.
HT_1.set_IsSpecAttached(True)
HT_1.set_AttachedSpecId(SP_1.Name)
HT_1.set_SpecVarType(SpecVarType.Source)
 
CL_1.set_IsSpecAttached(True)
CL_1.set_AttachedSpecId(SP_1.Name)
CL_1.set_SpecVarType(SpecVarType.Target)

In [12]:
sim.AutoLayout()

In [13]:
for i in range(3):
    errors = interf.CalculateFlowsheet4(sim)
    err_list = list(errors)
    print(f"Pass {i + 1} errors:", err_list)
    if not err_list:
        break
 
print("Spec status:", SP_1.get_Status())
print("Heater Heat Added:", HT_1.get_DeltaQ())
print("Cooler Heat Removed:", CL_1.get_DeltaQ())

Pass 1 errors: []
Spec status: 
Heater Heat Added: 100.0
Cooler Heat Removed: 100.0


In [14]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/21_Spec_Block/21_Spec_Block_Automation.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)